## 5. Iterators, Generators & Context Managers

This section covers two features that look unrelated but share the same idea: **special methods that let your objects plug into Python's syntax**.

* **Iteration** (every `for` loop) runs on the **iterator protocol**: `__iter__` and `__next__`.
* **Generators** are the easy way to create iterators, and they let you produce data *on demand*, which is essential when the data does not fit in memory.
* **The `with` statement** runs on the **context manager protocol**: `__enter__` and `__exit__`. It guarantees that resources (files, connections...) are released, even when something goes wrong.

### 5.1 Iterables & Iterators

#### **Iteration Fundamentals**

Most times, when writing a computer program, one may need to repeat some blocks of code multiple times. The main approaches followed are either **repeating** the target code as much as needed; or using a **loop** that (*magically?*) runs as many times as needed. 

Of course, it is not hard to see how the first one is just inefficient, plus hard to maintain. Using a loop is much better, specially for large programs: solves the problem and avoids the maintanability issue.

Everyone has seen the two main types of loops, in any programming language: **while** and **for.**

- The **while** loop runs as long as the loop-continuation condition remains `True`. This is known as **indefinite iteration:** executing some code a *potentially* undefined number of times.
  ```python
  while (True):
    ...
  ```
- The **for** loop runs for a predefined number of times, which is specially useful when needing to traverse over the items in a data structure 1-by-1. This is known as **definite iteration.**
  ```python
  for item in items:
    ...
  ```

In Python, if your iteration process requires going through the values or items in a data collection one item at a time, **iterators** will be needed.

##### **Iterable vs Iterator**

These two terms are often confused, but they name two different roles:

* An **iterable** is an object that *can be traversed*: when asked, it is able to give you an iterator. Lists, tuples, strings, dictionaries, sets, `range` objects and files are all iterables.
* An **iterator** is the object that *performs* the traversal: it keeps track of where it is, and produces the next element each time it is asked. When there are no elements left, it says so.

In other words, an **iterator** is an object that allows you to iterate over collections of data: lists, sets, dicts, etc. Python iterators implement the iterator **design pattern:** the iterator is used to traverse a **container** and access its elements.

They are in charge of: 

  - **Returning** the data from the container **one** item at a time.
  - **Keeping track** of the current and visited items.
  
And for that, iterators must implement the **iterator protocol.**

##### **The Iteration Protocol**

A Python object is considered an iterator when it implements two **special** methods known as the iterator protocol:

| Method | Implemented by | Description |
|---|---|---|
| **`.__iter__()`** | Iterables **and** iterators | Returns an iterator object. An iterable returns a **new** iterator; an iterator returns **itself.** Called to initialize it. |
| **`.__next__()`** | Iterators only | Called to **iterate** over the iterator. Returns the next element in the container. |

These methods are rarely called directly. Instead, two built-in functions are used:

* **`iter(obj)`** calls `obj.__iter__()` and gives you an iterator.
* **`next(iterator)`** calls `iterator.__next__()` and gives you the next element.

The `.__iter__()` method of an iterator usually returns `self`, holding a [reference to the current object](#41-class-fundamentals): the iterator itself. Most of the time it will look like: 

```python
def __iter__(self):
    return self
```
The only responsibility of `.__iter__()` is to return an iterator object. Remember that this object/instance must define still a **`.__next__()`** method.

The most generic use case of a Python iterator is to allow iteration over a **stream** of data or a **container data structure.** They support every operation that requires iteration: `for` loops, comprehensions, unpacking (`*args`, `**kwargs`)...

##### **`iter()`, `next()` and `StopIteration` by Hand**

Let's do manually what a `for` loop does automatically:

In [ ]:
numbers = [1, 2, 3]    # Or any iterable

it = iter(numbers)     # The list (an iterable) gives us an iterator
print(it)              # <list_iterator object at ...>

print(next(it))        # 1
print(next(it))        # 2
print(next(it))        # 3

try:
    next(it)           # No elements left -> StopIteration is raised
except StopIteration as e:
    print("StopIteration raised:", repr(e))

**`StopIteration` is not an error** in the usual sense: it is the *signal* an iterator uses to say "I'm done". `for` loops, `list()`, `sum()` and the rest catch it silently, so you only see it when calling `next()` by hand. (`try`/`except` is covered in section 6; here it just catches that signal so the cell doesn't stop.)

`next()` also accepts a second argument: a **default value**, returned instead of raising `StopIteration`:

In [ ]:
it = iter(["only element"])
print(next(it, "nothing left"))   # 'only element'
print(next(it, "nothing left"))   # 'nothing left' -> no exception

Every built-in collection is iterable, and each one decides *what* its iterator produces. For objects that aren't iterable, `iter()` raises a `TypeError`:

In [ ]:
print(list(iter("hey")))                     # A string produces its characters
print(list(iter({"a": 1, "b": 2})))          # A dict produces its ckeys
print(list(iter({"a": 1, "b": 2}.items())))  # .items() produces (key, value) pairs
print(list(iter(range(3))))                  # A range produces its numbers

try:
    iter(42)
except TypeError as e:
    print("TypeError:", e)                   # 'int' object is not iterable

##### **How a `for` Loop Works Internally**

By now you have probably noticed that most container objects can be looped over using a `for` statement:

```python
for element in [1, 2, 3]:
    print(element)
for element in (1, 2, 3):
    print(element)
for key in {'one': 1, 'two': 2}:
    print(key)
for char in "123":
    print(char)
for line in open("myfile.txt"):
    print(line, end='')
```

When Python executes

```python
for item in iterable:
    do_something(item)
```

what it actually does is:

1. Call `iter(iterable)` **once**, to get an iterator.
2. Call `next()` on that iterator, and assign the result to `item`.
3. Run the body of the loop.
4. Repeat steps 2 and 3 until `next()` raises `StopIteration`, which ends the loop silently.

We can write our own `for` with a `while` loop to prove it:

In [ ]:
def manual_for(iterable, body):
    iterator = iter(iterable)          # Step 1
    while True:
        try:
            item = next(iterator)      # Step 2
        except StopIteration:          # Step 4: the end signal
            break
        body(item)                     # Step 3

manual_for(["a", "b", "c"], print)     # Behaves exactly like: for x in ["a", "b", "c"]: print(x)

The same protocol is behind everything that *consumes* an iterable: `list()`, `tuple()`, `set()`, `dict()`, `sum()`, `min()`, `max()`, `sorted()`, `str.join()`, comprehensions, unpacking (`a, b = pair`), `*args` in function calls and the `in` operator. That's why, when you make your own class iterable, all of them start working with it at once.

##### **Multiple Traversals vs Single Traversal**

This is the most important practical difference between both concepts:

* **Iterables** like lists can be traversed **as many times as you want**, because every `for` asks them for a **new** iterator, which starts from the beginning.
* **Iterators** can be traversed **only once**. They only move forward, and when they reach the end they are **exhausted** ("empty"), and stay that way. Calling `iter()` on an iterator returns **the same object**, not a fresh one.

In [ ]:
colors = ["red", "green"]
print(list(colors), list(colors))     # A list: works every time

it = iter(colors)
print(list(it))                       # ['red', 'green']
print(list(it))                       # [] -> exhausted

print(iter(colors) is iter(colors))   # False: the list creates a new iterator every time
print(iter(it) is it)                 # True: an iterator returns itself

Many built-in functions return iterators instead of lists: `map()`, `filter()`, `zip()`, `enumerate()`, `reversed()`, file objects... so they are single-use too.

| | **Iterable** (e.g. a `list`) | **Iterator** (e.g. `iter(a_list)`, `map(...)`) |
|---|---|---|
| Has `__iter__()` | Yes: returns a **new** iterator | Yes: returns **itself** |
| Has `__next__()` | No | Yes |
| Remembers a position | No | Yes |
| Number of traversals | Unlimited | One |

##### **Exhaustion Problems**

Exhausted iterators are a classic source of issues, because nothing fails: the second traversal just silently finds nothing. Some typical cases:

In [ ]:
# 1. Reusing the result of map / filter / zip
prices = map(float, ["1.5", "2.0", "3.25"])
print(sum(prices))                    # 6.75
print(max(prices, default="EMPTY"))   # EMPTY -> sum() already consumed everything

# 2. The "in" operator consumes the iterator up to the element it finds
it = iter([1, 2, 3, 4])
print(3 in it)                        # True... but 1, 2 and 3 have been consumed
print(list(it))                       # [4]

# 3. A function that traverses its argument twice
def percentages(values):
    total = sum(values)                                # 1st traversal
    return [round(100 * v / total) for v in values]    # 2nd traversal

print(percentages([20, 30, 50]))          # [20, 30, 50] -> A list: fine
print(percentages(iter([20, 30, 50])))    # []           -> An iterator: the 2nd traversal is empty

The usual fix is to convert the iterator into a list when the data is needed more than once (`x = list(x)`), or to create a new iterator for each traversal. A defensive function can detect that it received an iterator, because for an iterator `iter(x) is x`:

In [ ]:
def percentages(values):
    if iter(values) is values:      # True: It is an iterator -> store its values first
        values = list(values)
    total = sum(values)
    return [round(100 * v / total) for v in values]

print(percentages(iter([20, 30, 50])))

#### **Custom Iterators**

##### **`__iter__()` and `__next__()`**

To create an iterator, write a class with the two methods of the protocol:

* **`__iter__(self)`**: must `return self`. This makes the iterator usable wherever an iterable is expected, like a `for` loop (which always starts by calling `iter()`).
* **`__next__(self)`**: computes and returns the next element, or does `raise StopIteration` when the sequence is finished.

The iterator's **state** (where it is in the traversal) is stored in **instance attributes**, through `self`. Each call to `__next__` reads the state, updates it, and returns a value. That's how the iterator "remembers" its position between calls.

Using the two methods that make up the iterator protocol, you can write at least three different types of custom iterators. Iterators that can:

- Take a stream of data and yield data items as they appear in the **original data.**
- Take a data stream, **transform** each item, and yield **transformed items.**
- Take no input data, generating **new data** as a result of some computation to finally yield the **generated items.**

The first type is a **classic** iterator because it implements the original pattern. The rest take it further.

##### **Yielding the Original Data**

As an example of **classic** iterator, let us create one that will take a **sequence** data type and yield its items on demand:

In [ ]:
class SequenceIterator:
    def __init__(self, sequence):
        self._sequence = sequence  # Takes a sequence of values at instantiation
        self._index = 0            # Underscore (_) attributes because no access from outside is needed


    def __iter__(self):
        return self     # Returns current object: self, the iterator itself

    def __next__(self):
        if self._index < len(self._sequence):   # Allows to control iteration
            item = self._sequence[self._index]
            self._index += 1
            return item
        else:
            raise StopIteration

# In a for loop:
for item in SequenceIterator([0, 1, 2, 3]): # Testing the iterator object
    print(item)

The custom iterator works as expected, taking any sequence as argument and allowing to iterate over it.

Recalling the internal functioning of `for` loops from before, one can simulate it again as:

In [ ]:
# (Execute cell above first)
sequence = SequenceIterator([0, 1, 2])

# Get the iterator object over the iterable
iterator = sequence.__iter__()
while True:
    try:
        # Retrieve next item
        item = iterator.__next__()
    except StopIteration:
        break
    else:
        # Loop's code block
        print(item)

##### **Transforming Input Data**

Say you want to write an iterator that takes any sequence data type and computes a **transformation** of each item, yielding the results on demand. For example:

In [ ]:
class SquareSequence:
    def __init__(self, sequence):
        self._sequence = sequence
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._index < len(self._sequence):
            square = self._sequence[self._index] ** 2
            self._index += 1
            return square
        else:
            raise StopIteration

for item in SquareSequence((0, 1, 2, 3)):
    print(item)

##### **Generating New Data**

You can also create custom iterators that generate a stream of new data from a given **computation** without taking a data stream as input.

As an example, an iterator producing the first `n` terms of the Fibonacci sequence:

In [ ]:
class FibonacciIterator:
    def __init__(self, n=5):    # No data input
        self._n = n             # How many terms to produce
        self._index = 0         # State: how many terms have been produced so far
        self._a = 0             # State: current term
        self._b = 1             # State: next term

    def __iter__(self):
        return self             # An iterator returns itself

    def __next__(self):
        if self._index < self._n:
            self._index += 1
            fib_number = self._a
            self._a, self._b = self._b, self._a + self._b
            return fib_number     # Yielding from no input, after computations
        else:
            raise StopIteration

fib = FibonacciIterator(10)
print(next(fib), next(fib), next(fib))   # Manual use
for term in fib:                         # The for loop continues from the current state
    print(term, end=" ")
print()
print(list(fib))                         # [] -> Exhausted from before

Because this last iterator follows the protocol, every tool that consumes iterables works with it, without writing anything else:

In [ ]:
print(list(FibonacciIterator(8)))
print(sum(FibonacciIterator(8)), max(FibonacciIterator(8)))
print(13 in FibonacciIterator(8))
a, b, c = FibonacciIterator(3)           # Unpacking
print(a, b, c)
print([term * 10 for term in FibonacciIterator(5)])

# We did not get exhaustion problems in any case here because all tools  create a new iterator every call.

##### ***Potentially* Infinite Iterators**

Python iterators can handle potentially infinite data streams, yielding values without reaching an end. For this, skip the `StopIteration` raise:

In [ ]:
class FibonacciInfIterator:
    def __init__(self):
        self._index = 0
        self._a = 0
        self._b = 1

    def __iter__(self):
        return self          # An iterator returns itself

    def __next__(self):      # Never raises the StopIteration exception
        self._index += 1
        self._a, self._b = (self._b, self._a + self._b)
        return self._a

fib = FibonacciInfIterator()
# for term in fib:     # Watch out: infinite loop
    # print(term)

Using infinite iterators will make `for` loops run endlessly. They will also cause functions that accept iterators (`sum()`, `max()`...) to never return. When using infinite iterators, you can make your code **hang.**

##### **Inheriting from `collections.abc`**

The `collections.abc` module includes an **ABC** called **`Iterator`.** You can use this ABC to create your custom iterators quickly. This class provides basic implementations for the `.__iter__()` method.

It also provides a `.__subclasshook__()` class method that ensures only classes implementing the iterator protocol will be considered subclasses of `Iterator`. For example, let's change a previous class to make it use the `Iterator` ABC:

In [ ]:
from collections.abc import Iterator

class SequenceIterator(Iterator):
    def __init__(self, sequence):
        self._sequence = sequence  
        self._index = 0

    def __next__(self):         
        if self._index < len(self._sequence):   # Allows to control iteration
            item = self._sequence[self._index]
            self._index += 1
            return item
        else:
            raise StopIteration

# In a for loop:
for item in SequenceIterator([0, 1, 2, 3]): # Testing the iterator object
    print(item)

No `.__iter__()` method is needed to be written because the **ABC** superclass already provides one with standard implementation. However, you **do** need to write your own **`.__next__()`** method. The features inherited from the `Iterator` ABC are useful when you are working with class hierarchies.

#### **Some Constraints**

By this point, we know that iterators allow to:

* Generate and yield a stream of data on demand
* Pause the iteration completely until the next value is required, which makes them ***lazy*.**
* Save memory by keeping only one value in memory at a given time
* Manage data streams of infinite or unknown size

But they have a few limitations that must be remembered when using them in your code. 

The first is that ***you cannot iterate over an iterator more than once*.** As an example, let's reuse the previous class:

In [ ]:
iter1 = SequenceIterator([0, 1, 2])
for number in iter1: # Consumed all the iterator items
    print(number)

for number in iter1: # No printing. Iterators can only be traversed once
    print(number)

Once you’ve consumed all the items from an iterator, that iterator is **exhausted.** An exhausted iterator’s only action is to raise a `StopIteration` exception, which immediately terminates any loop.

This leads to the second constraint: ***you cannot reset an exhausted iterator to start iteration again*.** If needed to iterate over the same data again, you must create a **new** iterator object:

In [ ]:
iter2 = SequenceIterator([0, 1, 2]) # NEW iterator
for number in iter2:
    print(number)

Another constraint of iterators is that they only define the `.__next__()` method, which gets the next item each time. There is no "`.__previous__()`" method or anything like that. This means that ***you can only move forward through an iterator. You cannot move backward*.**

Because iterators only keep one item in memory at a time, ***you cannot know their length or number of items,*** which is another limitation. You will only know its length once you have consumed all its items.

Finally, ***iterators do not allow indexing and slicing operations with the [ ] operator*:**

In [ ]:
iter1 = SequenceIterator([0, 1, 2])
print(iter1[1])
print(iter1[0:2])

#### **Python Iterables**

As defined before, an iterable is an **object** you can *iterate over*. Typically, using a `for` loop. Pure iterable objects usually hold the data themselves: Python built-in **container** types, for example: `list`, `tuple`, `dict`... They provide a stream of data that you can iterate over.

But, **iterators** are also iterable objects even if they do not hold the data themselves.

In Python, there are several valid definitions for an iterable:

##### **The Iterable Protocol**

Single special method already known: **`.__iter__()`** fulfills this protocol. Must return an **iterator object,** which usually does not coincide with `self` unless the iterable is also an iterator. For example:

In [ ]:
class Iterable:
    def __init__(self, sequence):
        self.sequence = sequence

    def __iter__(self):  # Makes this object iterable
        return SequenceIterator(self.sequence)  # Iterator object from before

for item in Iterable([0, 1, 2]): # Firstly calls iter(Iterable(...))
    print(item)

You cannot pass an iterable directly to the `next()` function because, in most cases, iterables do not implement the `.__next__()` method from the iterator protocol. This is intentional.

In [ ]:
letters = "abcd"
next(letters)  # TypeError: object is not an iterator

##### **Built-in `iter()` Function**

An iterable is an object that can be passed to the built-in `iter()` function to get an iterator from it. Internally, `iter()` falls back to calling `.__iter__()` on the target objects. Quick way to determine if an object is iterable.

In [ ]:
letters = "abcd"
print(iter(letters))

a = 5
print(iter(a))

##### **The Sequence Protocol**

Sequences are **container** data types that store items in consecutive **order.** Each item is quickly accessible through a **zero-based** index that reflects the item’s relative position in the sequence. Index + indexing operator (`[ ]`) to access individual items, as known already.

The sequence protocol consists of:

- **`.__getitem__(index)`** returns the item at that index in the underlying sequence. **`IndexError`** when out of range.
- **`.__len__()`** returns the length of the sequence.

The `.__getitem__()` method can be used for **iteration.** When an object implements it, you can loop over that object directly, even if the object does not define an `.__iter__()` method. Python calls `.__getitem__()` with consecutive integer indices, retrieving one item at a time. End of data when `IndexError` is raised.

In [ ]:
# Stack data structure example:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):  # Adds items on top of the stack
        self._items.append(item)

    def pop(self):  # Removes and returns items from the top of the stack
        try:
            return self._items.pop()
        except IndexError:
            raise IndexError("pop from an empty stack") from None

    # --Sequence Protocol--
    def __getitem__(self, index):  # __getitem__ provided: can iterate over Stack
        return self._items[index]

    def __len__(self):
        return len(self._items)

stack = Stack()
stack.push(0), stack.push(1), stack.push(2)

for item in iter(stack):
    print(item)

**Note:** Python dictionaries also implement `.__getitem__()` and `.__len__()`. However, they are considered **mapping data types** rather than sequences because the lookups use arbitrary **immutable** keys rather than integer indices.

##### **Iterables vs Iterators in Your Own Classes**

An iterator class like the one above is **single-use**. That is correct for an iterator, but wrong for an object that represents a *collection*: users expect to loop over a collection many times, and to nest loops over it.

The solution (the one built-in collections use: `list` and `list_iterator`) is to **split the two roles into two classes**:

* The **iterable** class holds the data. Its `__iter__()` returns a **new iterator object** every time it is called. It has no `__next__()`.
* The **iterator** class holds the state of one traversal, and implements `__iter__()` (returning `self`) and `__next__()`.

For example, a `Sentence` that can be traversed word by word:

In [ ]:
class Sentence:
    """Iterable: holds the data and creates a new iterator for every traversal."""
    def __init__(self, text):
        self.words = text.split()

    def __iter__(self):
        return SentenceIterator(self.words)


class SentenceIterator:
    """Iterator: remembers the position of ONE traversal."""
    def __init__(self, words):
        self.words = words
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.words):
            raise StopIteration
        word = self.words[self.index]
        self.index += 1
        return word

s = Sentence("Iterators remember their position.")
print(list(s))
print(list(s))                     # Works again: a new SentenceIterator was created

for w1 in Sentence("a b"):         # Nested loops: each loop has its own independent iterator
    for w2 in Sentence("a b"):
        print(w1 + w2, end=" ")

A common mistake is to write `__next__` directly in the collection class and return `self` from its `__iter__`: the first loop works, but then the collection becomes single-use, and nested loops break. **An iterable should not be its own iterator.**

##### **Comparing Iterators vs Iterables**

Iterables have an `.__iter__()` method that produce items on demand. Iterators implement an `.__iter__()` method that typically returns self and a `.__next__()` method that returns an item in every call.

You can conclude that ***all iterators are iterables*** because they meet the iterable protocol. However, ***not all iterables are iterators***—only those implementing the `.__next__()` method.

You may feel tempted to add a `.__next__()` method to a custom **iterable.** This addition will make it an iterable and an iterator at the same time. However, this practice is <u>**not recommended**</u> because it prevents multiple iterations over the underlying data.

With iterator objects, it is unlikely that you will get a new iterator every time, because their `.__iter__()` method typically returns `self`. This means that you will be getting the **same iterator** every time. In contrast, the `.__iter__()` method of an **iterable** will return a **new and different** iterator object every time you call it.

##### **Relation to OOP and Python's Protocols**

This is a clear example of [**polymorphism**](#45-polymorphism--abstraction) and **duck typing** in action. A `for` loop does not check whether your object is a `list`, or whether it inherits from some special class: it only checks whether the object *behaves* like an iterable, i.e. whether `iter()` works on it: "If it walks like a duck and quacks like a duck, it's a duck."

It also connects with the [**Data Model**](#43-dundermagic-special-methods--the-data-model): `__iter__` and `__next__` are dunder methods, just like `__len__` or `__getitem__`. In fact, Python keeps an older fallback: if a class has no `__iter__` but has a `__getitem__` that accepts 0, 1, 2..., `iter()` uses it, calling it with increasing indices until it raises an `IndexError`. `__iter__` is the modern, general way: it works with any source of data, not only index-based ones.

#### **Iteration Tools**

Python's built-ins include several functions designed around the iteration protocol. Those that produce a sequence of values return an **iterator** (so they are lazy and single-use).

##### **`enumerate()`**

**`enumerate(iterable, start=0)`** adds a counter to an iterable and returns it as an enumerate object, producing `(index, element)` pairs. It replaces the error-prone `for i in range(len(items)):` pattern:

In [ ]:
shopping = ["bread", "milk", "eggs"]

# Not Pythonic:
for i in range(len(shopping)):
    print(i, shopping[i], end=" ")
print("\n")

# Pythonic:
for position, item in enumerate(shopping, start=1):
    print(f"{position}. {item}", end=" ")
print("\n")

print(enumerate(shopping))          # The enumerate iterator object
print(list(enumerate(shopping)))    # [(0, 'bread'), (1, 'milk'), (2, 'eggs')]

##### **`zip()`**

**`zip(*iterables, strict=False)`** aggregates elements from **zero or more iterables,** returning an iterator that yields **tuples.**

| Argument | Description | Default Value |
|---|---|---|
| `*iterables` | Zero or more iterables such as lists, tuples, or strings to be zipped together. | Optional. Yields an empty iterator if omitted. |
| `strict`     | A Boolean that, if set to `True`, raises a `ValueError` if iterables have different lengths. | `False` |


Things to know:
* It stops as soon as the **shortest** iterable is exhausted, without warning. With `strict=True` (Python 3.10+), it raises a **`ValueError`** if the lengths differ.
* `dict(zip(keys, values))` is the classic way of building a dictionary from two lists.
* `zip(*rows)` does the inverse operation ("unzip"), which also transposes a matrix: the `*` unpacks the rows as separate arguments.

In [ ]:
letters = ["A", "B", "C"]
numbers = [1, 5, 2]

print(zip(letters, numbers))

for l, n  in zip(letters, numbers):
    print(f"{l}: {n}")

print(dict(zip(letters, numbers)))

print(list(zip(letters, [7, 4])))          # 'C' is silently dropped
try:
    list(zip(letters, [7, 4], strict=True))
except ValueError as e:
    print("ValueError:", e)

pairs = [("D", 0), ("E", 3)]
letters, numbers = zip(*pairs)                 # Unzip
print(letters, numbers)

matrix = [[1, 2, 3], [4, 5, 6]]
print(list(zip(*matrix)))                   # transpose: [(1, 4), (2, 5), (3, 6)]

##### **`reversed()`**

**`reversed(sequence)`** returns an iterator that walks a sequence **from the end to the beginning**, without copying it. It needs an object that knows its length and supports indexing (`__len__` + `__getitem__`), or that implements `__reversed__`. It does **not** work with iterators: they can only move forward, and they do not know where their end is.

Do not confuse it with `list.reverse()` (reverses the list in place, and returns `None`) or with `sequence[::-1]` (creates a new, reversed copy).

In [ ]:
backwards = reversed(range(1, 4))
for item in backwards:
    print(item)
print(backwards, list(backwards))       # <range_iterator ...> [] (exhausted)
print("".join(reversed("xyz")))         # 'zyx'

letters = ["a", "b", "c"]
print(letters.reverse(), letters)       # None ['c', 'b', 'a'] -> modified in place
print(letters[::-1], letters)           # A reversed copy; the original doesn't change

try:
    reversed(iter([1, 2, 3]))
except TypeError as e:
    print("TypeError:", e)

##### **`any()` and `all()`**

* **`any(iterable, /)`** returns `True` if **at least one** element in an iterable is truthy.
* **`all(iterable, /)`** returns `True` if **every** element in an iterable is truthy.
* Edge case: `any([])` is `False` (there is not any truthy element), and `all([])` is `True` (no element breaks the rule).

They make validations very readable, and they **short-circuit**: they stop traversing as soon as they know the answer (`any` at the first truthy element, `all` at the first falsy one). With an iterator, the remaining elements are not even computed.

(The `g >= 5 for g in grades` inside the calls below is a *generator expression*. For now, read it as a list comprehension without the brackets.)

In [ ]:
any([False, False, True])

grades = [8.5, 4.0, 9.2, 6.0]
print(all(g >= 5 for g in grades))     # Did everybody pass?  False
print(any(g >= 9 for g in grades))     # Is there a 9 or more? True
print(any([]), all([]))                # False True

def passed(grade):
    print(f"  checking {grade}")
    return grade >= 5

print(all(map(passed, grades)))        # Stops at 4.0: 9.2 and 6.0 are never checked

##### **`map()` and `filter()`**

* **`map(function, iterable, /, *iterables, strict=False))`** applies `function` to each element. Allows you to apply a function to each item of one or more iterables, producing an **iterator** that yields transformed items. By default, iteration stops when the **shortest** iterable is exhausted, and you can set `strict=True` to require equal lengths instead.
  
* **`filter(function, iterable)`**  returns an **iterator** that yields only the elements for which the filtering function returns a **truthy** value. With `None` as the function, it keeps the truthy elements themselves.

Both return **lazy iterators**: they compute nothing until they are consumed, and can be consumed only once. A comprehension usually expresses the same thing more readably; `map`/`filter` shine when the function already exists.

In [ ]:
raw = "10 25 3 47"
map_iter = map(int, raw.split())
print(map_iter)
numbers = list(map_iter)                    # Convert each piece to int. Produces iterator
print(numbers)
print(list(map(lambda a, b: a + b, [1, 2], [10, 20, 30])), "\n")   # Two iterables -> two arguments

filter_iter = filter(lambda n: n > 5, numbers)
print(filter_iter)
filtered = list(filter_iter)
print(filtered)
print(list(filter(None, ["", "x", None, 0, "y"])), "\n")       # Keeps only the truthy values

squares = map(lambda n: n**2, numbers)
print(squares)               # Nothing computed yet: just a map object
print(list(squares))         # Computed now...
print(list(squares), "\n")         # ...and only once

print([n**2 for n in numbers], [n for n in numbers if n > 5])   # The comprehension equivalents

##### **Introduction to [`itertools`](#the-itertools-module)**

The standard library module **`itertools`** contains many more iteration tools. Four important to start with are:

* **`count(start=0, step=1)`**: make an **iterator** that returns **evenly** **spaced** values beginning with start. Counts forever: `start`, `start + step`, `start + 2*step`... Can be used with `map()` to generate consecutive data points or with `zip()` to add sequence numbers.

In [ ]:
from itertools import count 

counter = count(start=1, step=1)  # Infinite count() iterator
print(counter, "\n")

zip_counter = zip(counter, ["a", "b", "c"])  # Zip: stops count() as soon as the other iterable is traversed
print(zip_counter, ": ", list(zip_counter), "\n")

squares_counter = map(lambda x: x**2, counter)
print(squares_counter, ": ", end="")
for _ in range(5):
    print(next(squares_counter), end=" ")


* **`cycle(iterable)`**: make an **iterator** returning elements from the **iterable** and saving a copy of each. When the iterable is exhausted, return elements from the saved copy. Repeats **indefinitely.** Repeats elements of an iterable forever.

In [ ]:
from itertools import cycle

cycler = cycle(["a", "b", "c"])  # Infinite cycle() iterator
print(cycler, "\n")

zip_cycler = zip(cycler, [1, 2, 3, 4, 5, 6]) # Again: stops cycle() as soon as the other iterable is exhausted.
print(zip_cycler, ": ", list(zip_cycler), "\n")

* **`chain(*iterables)`**: make an **iterator** that returns elements from the first iterable until it is **exhausted,** then proceeds to the next iterable, until all of the iterables are exhausted. Traverses several iterables one after the other, as if they were **one.**

In [ ]:
from itertools import chain

chainer = chain([1, 2], ["a", "b"], (True, False))
print(chainer, ": ", list(chainer))

* **`islice(iterable, stop)`** or **`islice(iterable, start, stop[, step])`**: make an **iterator** that returns selected elements from the iterable. Works like sequence **slicing** but does not support negative values for *start*, *stop*, or *step*. 

    Slicing (`[start:stop:step]`) for any iterable, **including iterators,** which do **not** support **`[]`**.

In [ ]:
from itertools import islice

counter = count(start=0, step=5)
slicer1 = islice(counter, 3)              # Returns elements 0, 1 and 2
print(slicer1, ": ", list(slicer1), "\n")

slicer2 = islice(counter, 2, 10, 2)       # Returns elements 2:10, in step of 2: 2, 4, 6, 8. 
print(slicer2, ": ", list(slicer2), "\n") # Note: we already traversed some elements of counter iterator.

In [ ]:
from itertools import count, cycle, chain, islice

print(list(islice(count(100, 10), 5)))            # First 5 values of an infinite counter
print(list(zip(count(1), ["x", "y", "z"])))       # Zip stops at the shortest iterable
print(list(islice(cycle(["day", "night"]), 5)))   # Alternating/repeating forever, cut at 5
print(list(chain([1, 2], (3, 4), "ab")), "\n")          # One iterable after the other

it = iter(range(10))
print(list(islice(it, 2, 5)))                     # kike [2:5], but on an iterator

Remember: `count` and `cycle` are **infinite**: always combine them with something that stops them, like `islice`, `zip` or a `break`.

### 5.2 Generators

#### **Generator Fundamentals**

##### **Definition**

Writing an iterator class can take a lot of ceremony: `__iter__`, `__next__`, the state in attributes, `raise StopIteration`... A **generator** is a much simpler way of creating an iterator: you write a normal function, but use the keyword **`yield`** to produce the values.

* A function that contains `yield` anywhere in its body is a **generator function**.
* Calling it **does not run its body**: it returns a **generator object**.
* A generator object **is** an iterator: it has `__iter__` and `__next__`, which Python writes automatically.

Generator functions are a special kind of function that return a ***lazy* iterator.** These are objects that you can **loop** over like a list. But, unlike lists or other containers, *lazy* iterators do **not** store their contents in memory.

In [ ]:
def three_greetings():
    yield "hello"
    yield "hola"
    yield "ciao"

gen = three_greetings()     # Nothing runs here
print(gen)                  # <generator object () at ...>
print(next(gen))            # First item
print(next(gen))            # Second item
print(next(gen))            # ...
print(next(gen, "-- finished --"), "\n")

print(list(three_greetings()))                       # A NEW generator, consumed by list()
print(hasattr(gen, "__next__"), iter(gen) is gen)    # It really IS an iterator

In short, generators are a simple and powerful tool for creating iterators. They look and act like regular functions but use the **`yield`** statement whenever they want to return data. For example:

In [ ]:
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

gen = infinite_sequence()
print(gen)

for _ in range(5):
    print(next(gen), end=" ")

`yield` indicates **where** a value is sent back to the caller, and unlike `return`, you do **not exit** the function after. Each time `next()` is called on it, the generator resumes where it left off (it remembers all the data values and which statement was last executed).

The **state** of the function is remembered (more on this later).

##### **`return` vs `yield`**

| | **`return`** | **`yield`** |
|---|---|---|
| What calling the function gives you | The returned value | A generator object (the body has NOT run yet) |
| How many values it produces | One | Any, one at a time |
| What happens to the function | It **ends**: its local variables are destroyed | It **pauses**: its local variables and position are kept |
| The next time | Calling it again starts from scratch | `next()` resumes right after the `yield` |

`return` can also appear inside a generator, with or without a value: it **finishes** the generator (see below).

##### **Understanding the `yield` Statement**

**`yield`** is a fairly simple statement. Its primary job is to control the flow of a generator function in a way that is similar to `return` statements.

When you call a generator function or use a generator expression, you return a special **iterator** called a generator. You can assign this generator to a variable in order to use it. When you call special methods on the generator, such as `next()`, the code within the function is executed up to `yield`.

When the `yield` statement is **hit,** the program **suspends** function execution and **returns the yielded value** to the caller. **`return`** **stops** function execution.

When the function is suspended, its state is **saved.** This allows you to resume function execution whenever you call one of the generator’s methods. For example:

In [ ]:
def multi_yield():
    yield_str = "This will print the first string"
    yield yield_str
    yield_str = "This will print the second string"
    yield yield_str

mygen = multi_yield()
print(next(mygen))
print(next(mygen))
print(next(mygen))  # Exhausted generator (StopIteration)

Another example: rebuilding one of the previous custom iterators as a **generator function:**

In [ ]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

fib_gen = fibonacci(5)
print(fib_gen)
for i in fib_gen:
    print(i, end=" ")
print("\n")

print(list(fibonacci(5)))
print(list(fibonacci(5)) == list(FibonacciIterator(5)))   # Identical behaviour

##### **Execution State and Resuming with `next()`**

A generator is a function that can be **paused and resumed**. Each `next()` runs the body from where it was up to the next `yield`: the yielded value is what `next()` returns, and the function freezes there, with all its local variables intact, until the following `next()`.

The function `inspect.getgeneratorstate()` shows the state of a generator at any moment:

* `GEN_CREATED`: created, but the body has not started yet.
* `GEN_RUNNING`: currently executing (only visible from inside the generator).
* `GEN_SUSPENDED`: paused at a `yield`.
* `GEN_CLOSED`: finished.

In [ ]:
from inspect import getgeneratorstate

def traced():
    print("    [body] start")
    x = 10
    yield x
    print(f"    [body] resumed, x is still {x}")   # Local variables survive the pause
    x += 5
    yield x
    print("    [body] end of the function")

gen = traced()
print("state:", getgeneratorstate(gen))    # GEN_CREATED: the body has not printed anything yet

print("next ->", next(gen))
print("state:", getgeneratorstate(gen))    # GEN_SUSPENDED at the first yield

print("next ->", next(gen))
print("state:", getgeneratorstate(gen))    # GEN_SUSPENDED at the second yield

print("next ->", next(gen, "StopIteration (replaced by the default)"))
print("state:", getgeneratorstate(gen))    # GEN_CLOSED

##### **Finishing a Generator and `StopIteration`**

A generator stops when its body ends, or when it reaches a `return`. At that moment `next()` raises `StopIteration`, exactly like a hand-written iterator, and `for` loops stop silently. A finished generator stays finished: there is no way to restart it, you have to call the generator function again to get a new one.

If the generator does `return value`, that value does **not** become an element: it is **attached** to the exception, as `StopIteration.value`. `for` loops ignore it (`yield from`, later, can capture it).

In [ ]:
def until_stop(words):
    for word in words:
        if word == "STOP":
            return "found STOP"     # Finishes the generator early
        yield word

print(list(until_stop(["a", "b", "STOP", "c"])))   # ['a', 'b']: 'c' is never reached

gen = until_stop(["x", "STOP"])
print(next(gen))
try:
    next(gen)
except StopIteration as e:
    print("StopIteration, value =", repr(e.value))  # e.value == StopIteration.value. return's value attached to the exception
print(next(gen, "still finished"))

##### **Generators in Classes: `__iter__` as a Generator**

Generators also simplify iterable classes. If `__iter__` is written as a generator method, every call to it returns a **new** generator, so the class is automatically a reusable iterable, and no separate iterator class is needed. From the previous section:

In [ ]:
class Sentence:
    def __init__(self, text):
        self.words = text.split()

    def __iter__(self):              # A generator method: each call creates a new generator
        for word in self.words:
            yield word

s = Sentence("Generators make this much shorter")
print(list(s))
print(list(s))                       # Reusable, like before

#### **Generator Expressions**

##### **Syntax**

A **generator expression** is to a generator what a list comprehension is to a list: it allows you to quickly create a generator object. The syntax is the same, but with **parentheses** instead of square brackets:

```python
[x**2 for x in data if x > 0]   # List comprehension   -> a list, built immediately
(x**2 for x in data if x > 0)   # Generator expression -> a generator, values **on demand**
```

When a generator expression is the **only argument** of a function call, its own parentheses can be omitted: `sum(x**2 for x in data)`.

In [ ]:
data = [3, -1, 4, -1, 5]

squares_list = [x**2 for x in data if x > 0]
squares_gen = (x**2 for x in data if x > 0)

print(squares_list, "\n")         # The resulting list

print(squares_gen)           # Generator object
print(next(squares_gen))     # Yield an item
print(list(squares_gen), "\n")     

print(sum(x**2 for x in data if x > 0))           # No extra parentheses needed
print(", ".join(str(x) for x in data))
print(max(len(w) for w in ["lazy", "evaluation"]))

You can create them **without** building and storing the entire object in **memory** before iteration. 

##### **Lazy vs Eager Evaluation**

* A list comprehension uses **eager** **(immediate)** evaluation: all the elements are computed **when the line runs**, and stored.
* A generator expression uses **lazy** evaluation: each element is computed **only when it is requested**, and then it is gone.

The difference becomes visible with a function that announces every computation:

In [ ]:
def expensive(x):
    print(f"  Computing {x}")
    return x * 100

print("List comprehension:")
eager = [expensive(x) for x in range(3)]      # Everything is computed right now
print("List created:", eager, "\n")

print("Generator expression:")
lazy = (expensive(x) for x in range(3))       # Nothing is computed yet
print("Generator created")
for value in lazy:                            # Each value is computed just before it is used
    print("  Using", value)

##### **Memory: Advantages**

A generator stores only its **current state,** not the values, so its size does not depend on how many elements it will produce. Compare one million computations:

In [ ]:
import sys

million_list = [x**2 for x in range(1_000_000)]
million_gen = (x**2 for x in range(1_000_000))

print(f"List: {sys.getsizeof(million_list)} bytes")   # ~8 MB
print(f"Generator: {sys.getsizeof(million_gen)} bytes")    # ~200 bytes, whatever the size

print(sum(million_list) == sum(million_gen))                    # Same result

(`sys.getsizeof()` of a list only counts its internal array of references, not the integer objects it points to, so the real difference is even bigger.)

Other advantages of laziness:

* **The first results are available immediately**, without waiting for all the data to be processed.
* **Work can be avoided**: if the consumer stops early (`break`, `any()`, `next()`), the remaining values are never computed.
* **Huge or infinite sequences** become possible. This is key in data science: big, GB-order files can be processed line by line with constant memory, instead of being loaded completely.

##### **Limitations**

Generators are not a free replacement for lists:

In [ ]:
gen = (x * 2 for x in range(5))

try:
    len(gen)                       # A generator does NOT know how many elements it will produce
except TypeError as e:
    print("TypeError:", e)

try:
    gen[0]                         # No indexing or slicing
except TypeError as e:
    print("TypeError:", e, "\n")

print(gen)                         # Printing it does not show the values
print(list(gen))                   # [0, 2, 4, 6, 8]
print(list(gen))                   # [] -> single use, exhausted, like every iterator

So, in the end:

| Use a **list comprehension** when... | Use a **generator expression** when... |
|---|---|
| You need the results several times | You traverse the results only once |
| You need `len()`, indexing, slicing or sorting | You pass the results straight to `sum`, `max`, `any`, `join`, a `for`... |
| The data is small | The data is large, or potentially infinite |
| You want to inspect or debug the values easily | You want to stop early or save memory |

Also, if you are going to consume *all* the values anyway and the data is small, a list comprehension is often slightly faster: laziness has a small cost per element.

#### **Advanced Generators**

##### **`yield from`**

**`yield from iterable`** produces, one by one, all the elements of **another iterable or generator.** In other words, the generator **delegates** part of its work:

```python
yield from iterable

# is (basically) equivalent to:
for item in iterable:
    yield item
```

In [ ]:
def letters_then_numbers():
    yield from "abc"          
    yield from range(3)       
    yield "end"

print(list(letters_then_numbers()))

def letters_then_numbers2():  # Equivalent to the generator function above
    for letter in "abc":
        yield letter
    for n in range(3):
        yield n
    yield "end"

print(list(letters_then_numbers2()))

It is especially useful in **recursive generators**, where a generator delegates to another call of itself. For example, walking a tree of folders represented with nested dictionaries (a folder is a `dict`, a file is a key with value `None`), producing the path of every file:

In [ ]:
project = {
    "README.md": None,
    "src": {
        "main.py": None,
        "utils": {"io.py": None, "maths.py": None},
    },
    "tests": {"test_main.py": None},
}

def walk(tree, prefix=""):
    for name, content in tree.items():
        path = f"{prefix}/{name}" if prefix else name
        if isinstance(content, dict):
            yield from walk(content, path)   # Delegate the sub-folder to a recursive call

          # Equivalent: 
          # for p in walk(content, path):
          #   yield p
        else:
            yield path

for path in walk(project):
    print(path)

Without `yield from`, the recursive line would be `for p in walk(content, path): yield p`. Besides, `yield from` is an expression that **evaluates to the `return` value** of the sub-generator (and later: it also forwards `send()`/`close()` to it):

In [ ]:
def numbered(items):
    for i, item in enumerate(items, start=1):
        yield f"{i}. {item}"
    return len(items)                       # The return value of the sub-generator...

def report(items):
    count = yield from numbered(items)      # ...is captured here
    yield f"({count} items)"

print(list(report(["eggs", "milk"])))

##### **Chaining Generators and Data Pipelines**

Since generators receive iterables and are iterables themselves, they can be **chained:** the output of one is the input of the next. Each one does a single small job, like the stations of an assembly line:

```
read  →  parse  →  filter  →  transform  →  consume
```

The key property: data flows through the whole chain **one element at a time**. The first element goes all the way to the end before the second one is even read, so memory usage stays constant no matter how large the input is. A trace makes it visible:

In [ ]:
def source(values):
    for v in values:
        print(f"Read {v}")
        yield v

def keep_even(numbers):
    for n in numbers:
        if n % 2 == 0:
            print(f"  Kept {n}")
            yield n

def times_ten(numbers):
    for n in numbers:
        print(f"    Transformed {n} -> {n * 10}")
        yield n * 10

pipeline = times_ten(keep_even(source([1, 2, 3, 4])))   #Nothing runs yet
for result in pipeline:
    print(f"      Consumed {result}")

Notice the interleaving: `read 1`, `read 2`, `kept 2`, `transformed`, `consumed`, and only then `read 3`. With lists, *all* the elements would be read first, then all of them filtered, and so on, keeping every intermediate list in memory.

Data pipelines allow you to string together code to process large datasets or streams of data without maxing out your machine’s memory.

A more realistic example: processing a file of temperature readings. Here the file is simulated in memory with `io.StringIO`; with a real file, `open()` would be used in exactly the same way. The stages can be generator functions or generator expressions:

In [ ]:
import io

readings_file = io.StringIO(
    "sensor,celsius\n"
    "A,21.5\n"
    "B,error\n"
    "A,23.0\n"
    "\n"
    "C,-4.5\n"
    "B,19.0\n"
)

def read_lines(f):
    next(f)                              # Skip the header (a file is an iterator of lines)
    for line in f:
        yield line.strip()

def parse(lines):
    for line in lines:
        if not line:
            continue                     # Skip empty lines
        sensor, value = line.split(",")
        try:
            yield sensor, float(value)
        except ValueError:
            print(f"  (Skipping invalid reading: {line!r})")

def to_fahrenheit(readings):
    return ((sensor, c * 9 / 5 + 32) for sensor, c in readings)   # A generator expression as a stage

lines = read_lines(readings_file)
readings = parse(lines)
fahrenheit = to_fahrenheit(readings)
above_freezing = (r for r in fahrenheit if r[1] > 32)

for sensor, temp in above_freezing:
    print(f"{sensor}: {temp:.1f} °F")

Each stage is small, independent and easy to test, and it can be reused with any input (a list, a file, another generator). This is the most useful **generator pattern** in practice.

##### **Infinite Generators and How to Control Them**

A generator with `while True:` produces values forever, but it is safe: it pauses at each `yield`, so it only computes what the consumer asks for. It is the **consumer** who must decide when to stop:

In [ ]:
from itertools import islice, takewhile

def ids(prefix):
    number = 1
    while True:                      # Infinite...
        yield f"{prefix}-{number:04d}"
        number += 1                  # ...but it only advances when asked

order_ids = ids("ORD")
print(next(order_ids), next(order_ids))          # 1) Manually, with next()

for invoice in ids("INV"):                       # 2) With a break
    if invoice.endswith("3"):
        break
    print(invoice)

print(list(islice(ids("TMP"), 3)))               # 3) With islice: take the first n
print(list(zip(["Ana", "Luis"], ids("USR"))))    # 4) zip() stops at the shortest input

def powers_of_two():
    n = 1
    while True:
        yield n
        n *= 2

print(list(takewhile(lambda p: p < 100, powers_of_two())))   # 5) Takewhile: until a condition fails

**Never** pass an infinite generator to something that consumes everything: `list(...)`, `sum(...)`, `sorted(...)`, `max(...)`... would run until the computer runs out of memory.

##### **Optional: `send()` and `close()`**

*This part is a conceptual introduction, optional at this stage. It will make more sense when you study asynchronous programming.*

Generators have methods that let the caller interact with a paused generator:

* **`gen.send(value)`**: resumes the generator, like `next()`, but also **sends a value into it**: the paused `yield` expression evaluates to `value` (with `next()`, it evaluates to `None`). Data now flows *into* the generator, not only out of it.
* **`gen.close()`**: stops the generator at the paused `yield`. If the generator has a `try`/`finally`, the `finally` block runs, which allows cleaning up.
* (`gen.throw(exception)` raises an exception inside the generator, at the paused `yield`.)

Generators that receive values with `send()` are the historical origin of **coroutines**, the basis of `async`/`await`. A small example: a generator that remembers the maximum of all the values sent to it:

In [ ]:
from inspect import getgeneratorstate

def running_max():
    current_max = None
    while True:
        value = yield current_max          # hands out the current max, then waits for a value
        if current_max is None or value > current_max:
            current_max = value

tracker = running_max()
next(tracker)                 # "prime" it: advance to the first yield (required before send)
print(tracker.send(5))        # 5
print(tracker.send(3))        # 5
print(tracker.send(9))        # 9
tracker.close()
print(getgeneratorstate(tracker))   # GEN_CLOSED

In [ ]:
def lines_with_cleanup():
    print("  opening resource")
    try:
        yield "line 1"
        yield "line 2"
        yield "line 3"
    finally:
        print("  releasing resource")    # runs even if the consumer stops early

gen = lines_with_cleanup()
print(next(gen))
gen.close()                              # the consumer stops after one line -> finally runs

This `try`/`finally` around a `yield` will be the foundation of `@contextmanager` in 5.4.3.

### 5.3 Advanced Iteration Tools

#### **The `itertools` Module**

<small>(Some general knowledge of this module can be enough for now)</small>

`itertools` is a standard library module of fast, memory-efficient building blocks for iteration. All its functions receive iterables and return **lazy iterators,** combining nice with generators. Includes the ones listed before and other essential ones:

| Category | Function | Produces |
|---|---|---|
| Infinite | `count(start, step)` | `start`, `start + step`, ... forever |
| | `cycle(iterable)` | The elements of `iterable`, repeated forever |
| | `repeat(x, n)` | `x`, `n` times (forever without `n`) |
| Combining / cutting | `chain(*its)`, `chain.from_iterable(its)` | The iterables, one after another |
| | `islice(it, start, stop, step)` | A slice of any iterable |
| | `zip_longest(*its, fillvalue=)` | Like `zip`, but up to the longest |
| Accumulating | `accumulate(it, func)` | Running totals (or a running `func`) |
| Grouping | `groupby(it, key)` | Groups of **consecutive** elements with the same key |
| Combinatorics | `product(*its, repeat=)` | The Cartesian product (like nested loops) |
| | `permutations(it, r)` | Ordered arrangements of `r` elements |
| | `combinations(it, r)` | Unordered selections of `r` elements |

##### **`chain()`**

**`chain(*iterables)`** traverses several iterables **one after another**, as if they were one. Compare with `list1 + list2`, which copies everything and only works between lists. Accepts iterables of any type.

**`chain.from_iterable(iterable_of_iterables)`** does the same when the iterables are themselves inside another iterable. It is the standard way of **flattening one level** of nesting.

Alternate constructor for `chain()`. Gets chained inputs from a **single** iterable argument.

In [ ]:
from itertools import chain

morning = ["email", "meeting"]
afternoon = ("coding", "review")
print(list(chain(morning, afternoon, ["gym"])))     # list + tuple + list

for task in chain(morning, afternoon):              # Loop over both without copying them
    print(task, end=" | ")
print()

weeks = [["mon", "tue"], ["wed"], ["thu", "fri"]]
print(list(chain.from_iterable(weeks)))             # Flatten one level

##### **`islice()`**

**`islice(iterable, stop)`** or **`islice(iterable, start, stop[, step])`** is slicing for any iterable. It exists because iterators and generators do not support `[start:(step):stop]`. Differences from normal slicing:

* No negative indices (an iterator does **not** know where its end is).
* It **consumes** the iterator: the skipped elements are read and discarded, and the iterator is left right after the last element taken.
* It returns an iterator (lazy), not a list.

In [ ]:
from itertools import islice

def squares():
    n = 0
    while True:
        yield n * n
        n += 1

print(list(islice(squares(), 5)))           # Like [:5]
print(list(islice(squares(), 5, 10)))       # Like [5:10]
print(list(islice(squares(), 0, 20, 5)))    # Like [0:20:5]

rows = iter(["header", "row 1", "row 2", "row 3", "row 4"])
header = next(rows)
first_two = list(islice(rows, 2))           # Takes 2 elements...
rest = list(rows)                           # ...and the iterator continues after them
print(header, first_two, rest)

##### **`count()`, `cycle()` and `repeat()`**

* **`count(start=0, step=1)`**: an infinite counter (it also works with floats and negative steps).
* **`cycle(iterable)`**: repeats the elements in a iterable, forever (it keeps a copy of the elements from the first pass).
* **`repeat(obj, times)`**: `obj`, repeated `times` times, or forever. Useful to provide a constant argument to `map` or `zip`.

In [ ]:
from itertools import count, cycle, repeat, islice

print(list(islice(count(0, 0.25), 5)))              # floats work too
print(list(islice(count(10, -3), 4)))               # counting down

for number, team in zip(range(1, 7), cycle(["red", "blue", "green"])):
    print(f"player {number} -> team {team}")        # assign teams in rotation

print(list(repeat("ha", 3)))
print(list(map(pow, [2, 3, 4], repeat(3))))         # 2**3, 3**3, 4**3

##### **`zip_longest()`**

**`zip_longest(*iterables, fillvalue=None)`**: make an iterator that aggregates elements from zero or more of each iterables. If the iterables are of uneven length, missing values are filled-in with `fillvalue`. If not specified, fillvalue defaults to `None`. 
  
Like `zip()`, but it continues until the **longest** iterable is exhausted.

In [ ]:
import operator
from itertools import zip_longest, accumulate

characters = ["A", "B", "C"]
IDs = ["001"]
print(list(zip(characters, IDs)))                         # Stops at the shortest
print(list(zip_longest(characters, IDs, fillvalue="-")))  # Keeps everyone

##### **`accumulate()`** 

**`accumulate(iterable, func=operator.add, initial=None)`:** make an iterator that returns **accumulated** sums or accumulated results from other binary functions. `function` defaults to addition. The function should accept two arguments, an accumulated total and a value from the iterable.

Produces **running results.** By default these are cumulative sums, but any two-argument function can be used (running maximum, running product...). With `initial`, it starts from a given value.

In [ ]:
daily_sales = [120, 80, 200, 50]
print(list(accumulate(daily_sales)))                    # Running total
print(list(accumulate(daily_sales, max)))               # Best day so far
print(list(accumulate([1, 2, 3, 4, 5], operator.mul)))  # Running product: factorials
print(list(accumulate(daily_sales, initial=1000)))      # Starting from an initial balance

##### **`groupby()`**

**`groupby(iterable, key=None)`:** make an iterator that returns consecutive **keys** and **groups** from the iterable. `key` is a function computing a key value for each element. If not specified or is `None`, key defaults to an identity function and returns the element unchanged.

Produces `(key, group)` pairs, where `group` is an iterator over **consecutive** elements that share the same key (`key(element)`, or the element itself if no `key` is given).

The word **consecutive** is what matters: `groupby` does **not** look at the whole data, only at neighbours. Equal keys that are not next to each other produce separate groups. That's why the data is almost always **sorted by the same key** first.

In [ ]:
from itertools import groupby

print([(k, list(g)) for k, g in groupby("aaabccaa")])   # 'a' appears in two separate groups

animals = ["cat", "cow", "dog", "duck", "cobra", "dove"]
for letter, group in groupby(animals, key=lambda a: a[0]):
    print(letter, list(group))           # Unsorted: the 'c' and 'd' groups get split
print("---")
for letter, group in groupby(sorted(animals), key=lambda a: a[0]):
    print(letter, list(group))           # Sorted: exactly one group per letter

**Careful:** each `group` is consumed as `groupby` moves on to the next key. Use every group **inside the loop** (converting it to a list there if needed). If the groups are stored and read later, they will be empty.

##### **`product()`**

**`product(*iterables, repeat=1)`:** **Cartesian product** of input iterables. Every possible combination taking one element from each iterable. It is equivalent to **nested `for` loops**, and replaces them with a single, flat loop. 

To compute the product of an iterable with **itself,** specify the number of repetitions with the optional `repeat` keyword argument.

In [ ]:
from itertools import product

sizes = ["S", "M", "L"]
colors = ["black", "white"]

# Nested loops...
for size in sizes:
    for color in colors:
        print(size, color, end=" | ")
print()

# ...are the same as a single loop over product()
for size, color in product(sizes, colors):
    print(size, color, end=" | ")
print()

cards = [rank + suit for suit, rank in product("SH", ["A", "K", "Q"])]
print(cards)

print(["".join(bits) for bits in product("01", repeat=3)])   # All the 3-bit strings

##### **`combinations()` and `permutations()`**

Both return successive **`r`** elements from an iterable:

* **`permutations(iterable, r=None)`**: **order matters**, so `(A, B)` and `(B, A)` are different results. If `r` is `None`, then it defaults to the length of the iterable.
* **`combinations(iterable, r)`**: **order does NOT matter**, so only one of `(A, B)` and `(B, A)` appears.
* **`combinations_with_replacement(iterable, r)`**: like `combinations`, but the same element can be picked more than once.

In [ ]:
import math
from itertools import permutations, combinations, combinations_with_replacement

examples = ["A", "B", "C", "D"]

triplets = list(permutations(examples, 3))     # Order matters
print(len(triplets), math.perm(4, 3))         # Same length
print(triplets, "\n")

pairs = list(combinations(examples, 2))       # Pairs of teammates: order does not matter
print(len(pairs), math.comb(4, 2))           # Same length
print(pairs, "\n")

print(list(combinations_with_replacement(["vanilla", "chocolate"], 2)), "\n")   # 2 scoops, repeats allowed

print([p for p in permutations("abc")])   # Every ordering

#### **`collections.abc`: Iterable, Iterator, Generator**

The module `collections.abc` defines three **abstract base classes** (ABCs) that represent Python's protocols:

| ABC | Abstract methods (required) | Methods provided for free |
|---|---|---|
| `Iterable` | `__iter__` | - |
| `Iterator` | `__next__` | `__iter__` (returning `self`) |
| `Generator` | `send`, `throw` | `__next__`, `close`, `__iter__` |

They form a hierarchy: `Generator` is a subclass of `Iterator`, which is a subclass of `Iterable`:

```
Iterable  ←  Iterator  ←  Generator
```

In words: **every generator is an iterator, and every iterator is an iterable.**

##### **Checking with `isinstance()`**

`isinstance(obj, Iterable)` returns `True` for any object with an `__iter__` method, even if its class **does not inherit from `Iterable`**. These ABCs look at the *structure* of the class (through a special hook called `__subclasshook__`) instead of its family tree. It is the formal version of *duck typing*: what matters is the methods, not the inheritance.

In [ ]:
from collections.abc import Iterable, Iterator, Generator

def gen_function():
    yield 1

objects = {
    "list": [1, 2],
    "dict": {"a": 1},
    "iter(list)": iter([1, 2]),
    "map(...)": map(str, [1]),
    "Sentence": Sentence("a b"),
    "FibonacciIterator": FibonacciIterator(3),
    "gen_function()": gen_function(),
    "(x for x in ...)": (x for x in "ab"),
    "42": 42,
}

print(f"{'object':20} {'Iterable':>9} {'Iterator':>9} {'Generator':>10}")
for name, obj in objects.items():
    print(f"{name:20} {isinstance(obj, Iterable)!s:>9} {isinstance(obj, Iterator)!s:>9} "
          f"{isinstance(obj, Generator)!s:>10}")

**One limitation:** `isinstance(obj, Iterable)` does not detect objects that are iterable only through the old `__getitem__` fallback. The only completely reliable test is to try `iter(obj)`:

In [ ]:
from collections.abc import Iterable

class OnlyGetitem:
    def __getitem__(self, i):
        if i >= 3:
            raise IndexError
        return i

obj = OnlyGetitem()
print(list(obj))                     # [0, 1, 2]: it IS iterable...
print(isinstance(obj, Iterable))     # False: ...but the ABC does not know it

def is_iterable(x):
    try:
        iter(x)
        return True
    except TypeError:
        return False

print(is_iterable(obj), is_iterable(42))

##### **Inheriting from the ABCs**

Inheriting explicitly from these ABCs is optional, but it has two benefits: you get the "free" methods (called *mixin methods*), and Python refuses to create instances of a subclass that forgot an abstract method. For example:

In [ ]:
from collections.abc import Iterator

class Countdown(Iterator):
    def __init__(self, start):
        self.current = start

    def __next__(self):              # __iter__ is inherited from Iterator
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

print(list(Countdown(5)))

class Broken(Iterator):              # Forgot __next__ -> no (sub)class instance creation allowed
    pass

try:
    Broken()
except TypeError as e:
    print("TypeError:", e)

##### **Use in Type Hints**

These ABCs are also used in **type hints**, to describe as generally as possible what a function accepts or returns. They can be subscripted with the type of the elements, as in `Iterable[type]`:

In [ ]:
from collections.abc import Iterable, Iterator

def average(values: Iterable[float]) -> float:    # Accepts lists, tuples, sets, generators...
    total = count = 0
    for v in values:                               # A single traversal: works with iterators too
        total += v
        count += 1
    return total / count

def evens(limit: int) -> Iterator[int]:            # A generator function returns an Iterator
    yield from range(0, limit, 2)

print(average([1, 2, 3]), average((x / 2 for x in range(5))), list(evens(7)))

Writing `Iterable[float]` instead of `list[float]` documents that the function only needs to loop over its argument once, so any iterable is welcome. For generator functions, `Iterator[int]` is the usual hint; `Generator[YieldType, SendType, ReturnType]` is used when `send()` or the `return` value matter.

### 5.4 Context Managers

#### **Context Manager Fundamentals**

##### **Definition**

Programs constantly use **resources** that must be **acquired** before using them and **released** afterwards:

| Resource | Acquire | Release |
|---|---|---|
| File | Open it | Close it (writing pending data to disk) |
| Database / Network connection | Connect | Disconnect |
| Temporary folder | Create it | Delete it |
| ... | ... | ... |

Sometimes, a program uses a given resource and does not release the associated memory when it no longer needs the resource. This kind of issue is called a *memory leak*. Properly managing these **external resources** is very important: it requires ***setup*** (acquiring) and ***teardown*** (releasing) phases.

A **context manager** is an object that encapsulates this pair of actions: it defines what happens when entering a block of code (setup, acquiring) and what happens when leaving it (teardown, releasing), **however** the block is left. 

In Python, a context manager is an object that allows you to handle a context where you **allocate** and **release** resources as needed.  It is used with the **`with`** statement. Working with files is probably the most common example of resource management in programming.

In Python, you can use a couple of general approaches to deal with resource management. You can wrap your code in:

1. A `try ... finally` construct
2. A `with` construct
   
When managing external resources in Python, you can use these constructs to handle **setup and teardown logic.**

##### **`try ... finally`**

This first approach is quite **generic** and allows you to provide setup and teardown code to manage any kind of resource. You can use a `try ... finally` construct to handle opening and closing files properly, for example:

In [ ]:
file = open("hello.txt", "w")

try:
    file.write("Hello, World!")     # Setup logic
finally:    
    file.close()   # File will be properly closed even if an exception occurs. finally always runs. Teardown logic

The `try` block from above can potentially raise exceptions. You can handle these exceptions with an `except` clause:

In [ ]:
file = open("hello.txt", "w")

try:
    file.write("Hello, World!")
except Exception as e:
    print(f"An error occurred while writing to the file: {e}")
finally:
    file.close()

Even though the try … finally construct works correctly, Python has a more compact and elegant solution: the `with` statement.

##### **Managing Resources by Hand vs `with`**

Writing to a file by hand:

```python
f = open("hello.txt", "w")
f.write("Hello World")    # If this line raises an exception...
f.close()                 # ...this line never runs: the file stays open
```

To guarantee the release even when errors happen, you need `try`/`finally` (the `finally` block always runs, whether the `try` block ends normally or with an exception):

```python
f = open("hello.txt", "w")
try:
    f.write("Hello World")
finally:
    f.close()             # Always runs
```

The `with` statement does exactly the same, with less code and with no chance of forgetting the release:

```python
with open("hello.txt", "w") as f:
    f.write("Hello World")
# Here the file is guaranteed to be closed
```

##### **The `with` Statement**

The Python `with` statement creates a **runtime context** that allows you to execute a code block under the control of a **context manager.** `with` makes the code clearer, safer and more readable.

Many classes and objects in the standard library support the `with` statement by implementing the **context manager protocol**, which are special methods. The general **syntax** of a `with` statement is as follows:

```python
with expression [as obj]:
    <block>
```

The context manager object results from evaluating the **expression** after the `with` keyword. In other words, `expression` must return an object that implements the context manager **protocol.**

This protocol consists of two special methods:

- **`.__enter__()`** is called by the `with` statement to **enter** the runtime context. <br>
- **`.__exit__()`** is called when the execution **leaves** the `with` code block.


Internally, the statement does:

1. `expression` is evaluated, and it produces a context manager.
2. The context manager's **setup** runs (call `.__enter__()` on it). Its result is assigned to `obj` (the `as` specifier is **optional**).
3. `block` runs.
4. The **teardown** runs when the block ends (call `.__exit__()` on it), no matter how: normally, with `return`/`break`/`continue`, or with an exception (which then keeps propagating, unless the context manager decides otherwise).

Redoing the previous example with this new construct:

In [ ]:
with open("hello.txt", mode="w", encoding="utf-8") as file:
    file.write("Hello, World!")

When running this `with`, `open()` returns a `io.TextIOBase` object, which support the context manager protocol, so `with` calls `.__enter__()` on it and assings the result to `file`. After writing, Python calls `.__exit__()`.

This statement is shorter than the previous construct and also **less** generic, since it only supports objects that implement the protocol.

It also supports **multiple** context managers:

```python
with A() as a, B() as b:
    <block>
```

It works like **nested** `with` statements, but without nesting. Useful when opening many files at a time:

In [ ]:
with open("input.txt") as in_file, open("output.txt", "w") as out_file:
    for line in in_file:
        out_file.write(line.upper())

So, `with` helps avoid resource leaks by making it almost impossible to forget to clean up, close, and release resources after you are done with them. Packs setup and teardown logic into a context manager and handles it.

#### **Class-Based Context Managers: The Context Manager Protocol**

##### **`__enter__()` and `__exit__()`**

Any object becomes a context manager by implementing two special methods:

* **`__enter__(self)`**: handles **setup** logic. Called when entering the `with` block. Whatever it **returns** is assigned to the variable after `as`. It often returns `self`, but it can return anything (or nothing).
* **`__exit__(self, exc_type, exc_value, traceback)`**: handles **teardown** logic. Called when leaving the block, **always**. Its three parameters describe the exception that ended the block:
    * If the block finished normally: `None`, `None`, `None`.
    * If an exception occurred: its class, the exception object itself, and its traceback (information about where it happened).

Internally, `with manager as name: block` is roughly equivalent to:

```python
name = manager.__enter__()  # "manager as name"
try:
    block
except BaseException as exc:
    if not manager.__exit__(type(exc), exc, exc.__traceback__):
        raise                      # The exception continues, unless __exit__ returned True
else:
    manager.__exit__(None, None, None)
```

A simple class-based context manager can be, for example:

In [ ]:
class HelloContextManager:
    def __enter__(self):
        print("Entering the context...")
        return "Hello, World!"

    def __exit__(self, exc_type, exc_value, exc_tb):
        print("Leaving the context...")
        print(f"{exc_type  = }")
        print(f"{exc_value = }")
        print(f"{exc_tb    = }")

with HelloContextManager() as hello:
    print(hello)

When the with statement runs, Python creates a new instance of the class and calls its `.__enter__()` method. Next, it runs the `with` code block (printing). When the execution flow exits the `with` block, Python calls `.__exit__()`.  If no exception occurs before, its three arguments are set to `None`. 

If one does occur: 

In [ ]:
with HelloContextManager() as hello:
    print(hello)
    hello[100]  # __exit__ called as soon as a exception occurs, like in this line.

Let's build a more realistic example: a context manager that can open files for writing directly.

In [ ]:
class WritableFile:
    def __init__(self, file_path):
        self.file_path = file_path

    def __enter__(self):
        self.file_obj = open(self.file_path, mode="w", encoding="utf-8")  # Setup. Acquire the file resource for use.
        return self.file_obj

    def __exit__(self, exc_type, exc, tb):
        if self.file_obj:
            return self.file_obj.close()  # Teardown. Release the previous acquired file object by closing it.

with WritableFile("hello.txt") as file:
    file.write("Hello, World!")  # "with" block 

Another example is a context manager that can measure execution time:

In [ ]:
import time

class Timer:
    def __init__(self, label="Block"):
        self.label = label
        self.elapsed = None

    def __enter__(self):
        self.start = time.perf_counter()
        return self                              # 'as t' receives the Timer itself

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}]: {self.elapsed:.4f} s")
        return False                             # Do not hide exceptions

with Timer("List of squares") as t:
    squares = [n**2 for n in range(1_000_000)]

print("Still available after the block:", round(t.elapsed, 4))

with Timer("Sum with a generator"):
    sum(n**2 for n in range(1_000_000))

##### **Exception Handling and Propagation**

When an exception happens inside the block, Python calls `__exit__` with the information about the exception, and then looks at what `__exit__` **returns**:

* **A falsy value** (`False`, or `None`): the exception **propagates** normally after the teardown, as if there were no `with`. Default behaviour if it does not return anything explicitly.
* **A truthy value** (`True`): any exception that occurs in the block is **suppressed** (swallowed). Execution continues at the next line **after** `with`, and the remaining lines of the block are **not** executed.

In both cases, the cleanup runs. The three possible scenarios, with a context manager that can be told whether to suppress or not:

In [ ]:
class Guard:
    def __init__(self, suppress):
        self.suppress = suppress

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("  __exit__: block finished normally")
        else:
            print(f"  __exit__: block failed with {exc_type.__name__}: {exc_value}")
        return self.suppress

print("1) No exception:")
with Guard(suppress=False):
    print("  Working...")
print("  After the \"with\"\n")

print("2) Exception, __exit__ returns False -> it propagates:")
try:
    with Guard(suppress=False):
        print("  Working...")
        raise KeyError("Missing key")
        print("  Never runs (exception raised -> __exit__ called)")
    print("  Never runs either: the exception left the \"with\" block")
except KeyError as e:
    print("  Caught outside:", repr(e), "\n")

print("3) Exception, __exit__ returns True -> it is suppressed:")
with Guard(suppress=True):
    print("  Working...")
    raise KeyError("Missing key")
    print("  Never runs: the rest of the block is skipped")
print("  Execution continues after the with")

Suppressing exceptions indiscriminately hides bugs. When a context manager suppresses exceptions, it should only do it for **specific, expected exception types**, checking `exc_type`:

In [ ]:
class IgnoreMissingKeys:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        # True only for KeyError (and its subclasses): anything else propagates
        return exc_type is not None and issubclass(exc_type, KeyError)

keys = {"key1": "value1"}

with IgnoreMissingKeys():
    print(keys["key1"])
    print(keys["key2"])    # KeyError -> suppressed
print("continues")


with IgnoreMissingKeys():
    1 / 0                    # ZeroDivisionError -> not suppressed

#### **Function-Based Context Managers: The `contextlib` Module**

##### **The `@contextmanager` Decorator**

Writing an entire context manager class can be a bit unnecessary for simple cases. If you decorate an appropriately coded generator function with **`@contextmanager`**, then you get a function-based context manager that automatically provides both `.__enter__()` and `.__exit__()`.

```python
from contextlib import contextmanager

@contextmanager
def func(args):
    # 1. Setup                     ->  __enter__
    try:
        yield value                # 2. with block
    finally:
    # 3. Teardown                  ->  __exit__
```

Before the `yield` statement, you have the **setup** section: acquiring the needed resources. When entering the context, everything until `yield` runs. After `yield`, you have the **teardown** section: releasing resources. This part runs at the end of the `with` block.

The `yield` statement optionally provides the object that will be assigned to the `with` target variable (`as ...`).

For example:

In [ ]:
from contextlib import contextmanager

@contextmanager
def hello_context_manager():
    # 1. Setup
    print("Entering the context...")      # -> __enter__

    yield "Hello, World!"                 # 2. with block
    # 3. Teardown
    print("Leaving the context...")       # -> __exit__

with hello_context_manager() as hello:
    print(hello)

This implementation and the one that uses the context management protocol are **equivalent.** The `@contextmanager` decorator reduces the boilerplate required to create a context manager. You just need to implement a **generator function** with a single `yield` that produces whatever you want `.__enter__()` to return.

Let's use this approach in a simple file writing problem:

In [ ]:
from contextlib import contextmanager

@contextmanager
def writable_file(file_path):
    try:
        file = open(file_path, mode="w")    # Setup: acquire the resource (file)
        yield file
    finally:
        file.close()                        # Teardown: release it

with writable_file("hello.txt") as file:
    file.write("Hello, World!")

Another example, rewriting the previous `Timer` class as a function-based context manager:

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(label="Block"):
    start = time.perf_counter()                  # Setup
    try:
        yield                                    # The with block runs during this pause
    finally:
        elapsed = time.perf_counter() - start    # Teardown/Cleanup
        print(f"[{label}]: {elapsed:.4f} s")

with timer("Sorting"):
    sorted(range(1_000_000, 0, -1))

If an exception happens inside the `with` block, it is **re-raised inside the generator, at the `yield` line**. Without `try`/`finally`, the code after the `yield` would be skipped, and the cleanup would never happen:

In [ ]:
@contextmanager
def unsafe():
    print("  Acquire")
    yield
    print("  Release")              # Skipped when the block fails

@contextmanager
def safe():
    print("  Acquire")
    try:
        yield
    finally:
        print("  Release")          # Always runs

for manager in (unsafe, safe):
    print(manager.__name__ + ":")
    try:
        with manager():
            raise RuntimeError("Failure inside the block")
    except RuntimeError:
        print("  RuntimeError propagated")

Since the [exception](#6-exception-handling) appears at the `yield`, it can be handled with a normal `try`/`except` around it:

* Catching it and **not** re-raising it → the exception is **suppressed** (like `__exit__` returning `True`).
* Catching it and doing `raise` → it is handled (logged, undone...) and then **propagated**.

The inventory transaction, rewritten with `@contextmanager`:

In [ ]:
import copy
from contextlib import contextmanager

@contextmanager
def inventory_transaction(inventory):
    backup = copy.deepcopy(inventory)
    try:
        yield inventory
    except Exception as e:
        inventory.clear()
        inventory.update(backup)
        print(f"  Rolled back because of: {e}")
        raise                          # Propagate the error after undoing the changes

stock = {"apples": 10}
try:
    with inventory_transaction(stock) as inv:
        remove_stock(inv, "apples", 4)
        remove_stock(inv, "apples", 40)
except ValueError:
    pass
print(stock)                           # {'apples': 10}

Another very common pattern: changing something temporarily, and restoring it afterwards even if the block fails:

In [ ]:
@contextmanager
def temporary_setting(settings, key, value):
    old_value = settings[key]
    settings[key] = value               # change
    try:
        yield settings
    finally:
        settings[key] = old_value       # always restore

settings = {"debug": False, "language": "en"}
with temporary_setting(settings, "debug", True):
    print("inside: ", settings)
print("outside:", settings)

##### **Class-Based vs Function-Based**

| | **Class** (`__enter__`/`__exit__`) | **`@contextmanager`** (generator) |
|---|---|---|
| Amount of code | More | Less |
| Setup and cleanup | In two separate methods | Together, around the `yield` |
| Exception information | Received as arguments of `__exit__` | Caught with `try`/`except` around the `yield` |
| Suppressing an exception | `return True` from `__exit__` | Catch it and don't re-raise it |
| State and extra methods | Natural: attributes and methods (e.g. `t.elapsed`) | Awkward: only local variables |
| Reusing the object | Can be designed to be reused | Each call creates a single-use context manager |
| Best for | Objects that *are* resources, or need state / an API | Simple setup/cleanup logic |

Rule of thumb: start with `@contextmanager`, and switch to a class when the context manager needs to keep state that the user reads afterwards, or to offer methods.

##### **`suppress()`**

<small>(Maybe not needed to know in detail, but worth mentioning)</small>

**`contextlib.suppress(*exceptions)`** is a ready-made context manager that ignores the given exception types. It replaces the `try: ... except X: pass` pattern, when ignoring the exception really is what you want:

In [ ]:
import os
from contextlib import suppress

# Instead of:
try:
    os.remove("file_that_does_not_exist.tmp")
except FileNotFoundError:
    pass

# Write:
with suppress(FileNotFoundError):
    os.remove("file_that_does_not_exist.tmp")

with suppress(KeyError, IndexError):     # several types at once
    print([1, 2, 3][10])
print("Continues normally")

##### **`closing()`**

<small>(Maybe not needed to know in detail, but worth mentioning)</small>

Some objects have a `close()` method but do not implement the context manager protocol (old libraries, some network objects...). **`contextlib.closing(obj)`** wraps them: entering returns `obj`, and exiting calls `obj.close()`:

In [ ]:
from contextlib import closing

class LegacyConnection:
    """An object with close(), but without __enter__/__exit__."""
    def query(self, sql):
        return f"Results of {sql!r}"

    def close(self):
        print("  Connection closed")

with closing(LegacyConnection()) as conn:
    print(conn.query("SELECT 1"))

##### **~~`redirect_stdout()`~~**

<small>(Not really needed right now)</small>

**`contextlib.redirect_stdout(target)`** temporarily sends everything written with `print()` to another file-like object, for example an `io.StringIO`, to capture it as a string. It's useful to capture the output of a function that prints instead of returning:

In [ ]:
import io
from contextlib import redirect_stdout

def chatty_function():
    print("Step 1 done")
    print("Step 2 done")
    return 42

buffer = io.StringIO()
with redirect_stdout(buffer):
    result = chatty_function()          # its prints go into the buffer

print("Result:", result)
print("Captured output:", buffer.getvalue().splitlines())

##### **~~`ExitStack` and `async with`~~**

Two more advanced tools for later:

<small>(WIP, maybe in the future if asynchronous programming is needed)</small>

### 5.5 Summary

**Iterables & iterators:** An *iterable* can produce iterators (`__iter__`); an *iterator* produces elements one at a time (`__next__`) and raises `StopIteration` at the end. Iterables can be traversed many times; iterators only once (exhaustion). A custom iterable should return a *new* iterator from `__iter__`.

**Generators:** A function with `yield` returns a generator: an iterator whose body pauses at each `yield` and resumes on `next()`, keeping its local state.  Generator expressions are lazy list comprehensions: constant memory, but single use and no `len()`. Generators can delegate with `yield from`, be chained into pipelines, and be infinite.

**Advanced iteration tools:** `itertools` provides lazy building blocks: `chain`, `islice`, `count`, `cycle`... `collections.abc` names the protocols: `Generator` is an `Iterator`, which is an `Iterable`, checkable with `isinstance()` without inheritance.

**Context managers:** `with` guarantees that resources are released, also when errors happen, replacing manual `try`/`finally`. The protocol: `__enter__` +  `__exit__`. `@contextmanager` builds one from a generator: setup, `yield` inside `try`, cleanup in `finally`. `contextlib` also provides other utiilities.

For more information: [Iterator Types (Python docs)](https://docs.python.org/3/library/stdtypes.html#iterator-types), [PEP 255 – Simple Generators](https://peps.python.org/pep-0255/), [`itertools`](https://docs.python.org/3/library/itertools.html), [`collections.abc`](https://docs.python.org/3/library/collections.abc.html), [PEP 343 – The "with" Statement](https://peps.python.org/pep-0343/), [`contextlib`](https://docs.python.org/3/library/contextlib.html).